# Детекция ботов по событиям куки

## Подход коротко

**Задача.** По событиям куки внутри суточного окна дать score: насколько вероятно, что это бот (сервис сбора данных). Метрика — precision при recall ≥ 0.7.

**Данные.** Сначала почистил события. Выкинул всё, что после окна: там утечка, у ботов таких событий в 4 раза больше, и там же вся капча. Удалил полные дубли строк, отсортировал по времени, привёл `platform` к трём значениям.

**Признаки поведения.** Считаются только по событиям самой куки в её окне:
- интервалы между событиями (у ботов 3–20 с, у людей 15–120 с);
- курсор (у ботов его часто нет или он прижат к левой части экрана);
- глубина листания выдачи и разнообразие объявлений и городов;
- порядок действий;
- User-Agent, разобранный на части.

На этом LightGBM даёт около 0.80.

**Признаки по объявлениям.** Разбирая ошибки, заметил, что боты смотрят в основном одни и те же объявления: ~500 объявлений дают 17% просмотров ботов и 2% просмотров людей. Отсюда три группы признаков:
1. популярность объявлений куки среди других кук, без меток;
2. доля ботов среди тех, кто смотрел эти объявления раньше: только по меткам прошлых дней;
3. средний скор «соседей» по объявлениям, через двухэтапную модель.

Всё считается только по данным, доступным к концу окна. С этими признаками метрика выросла до 0.90.

**Валидация.** Только по времени, потому что тест идёт сразу после train:
- 4 фолда с расширяющимся окном;
- отдельный сплит на 7 дней, как у теста.

Baseline из quickstart даёт 0.096, финальная модель 0.90 на фолдах и 0.88 на 7-дневном сплите.

Код разбит на два модуля, ноутбук их вызывает и показывает, что происходит по шагам:
- `features.py` — очистка событий и все признаки;
- `train.py` — модели и валидация. Тот же результат даёт `python train.py`.

In [1]:
import platform
import warnings

import lightgbm
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import roc_auc_score

import features as F      # очистка событий и признаки
import train as T         # модели и валидация

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

print("python", platform.python_version(), "| pandas", pd.__version__,
      "| lightgbm", lightgbm.__version__, "| sklearn", sklearn.__version__)

python 3.11.0 | pandas 3.0.6 | lightgbm 4.7.0 | sklearn 1.7.2


## 1. Данные

In [2]:
train, test, events = F.load("data")
meta = pd.concat([train, test], ignore_index=True)

print("train:", train.shape, " test:", test.shape, " events:", events.shape)
print("доля ботов в train:", round(train.target.mean(), 4))
print("train:", train.window_start_ts.min().date(), "-", train.window_start_ts.max().date(),
      "| test:", test.window_start_ts.min().date(), "-", test.window_start_ts.max().date())

train: (11091, 5)  test: (4909, 4)  events: (328905, 14)
доля ботов в train: 0.0811
train: 2026-04-06 - 2026-04-19 | test: 2026-04-20 - 2026-04-26


## 2. Что не так с событиями

Прежде чем считать признаки, проверил файл событий. Ниже каждая проблема с цифрами.

In [3]:
ev = events.merge(meta[["cookie_id", "window_start_ts", "window_end_ts", "target"]], on="cookie_id")
after = ev.event_ts >= ev.window_end_ts

# 1) события после окна: их нельзя использовать, и они сильно различаются между классами
print("событий после окна:", round(after.mean(), 3))
print(ev[ev.target.notna()].assign(after=after).groupby("target").after.mean().round(3)
      .rename("доля событий после окна"))

# 2) капча вся после окна -> внутри окна её нет вообще
cap = ev.event_name == "captcha_shown"
print("\ncaptcha_shown всего:", cap.sum(), " из них внутри окна:", (cap & ~after).sum())

# 3) полные дубли строк
print("\nполных дублей строк:", events.duplicated().sum())

# 4) порядок строк
print("строки отсортированы по времени:", events.event_ts.is_monotonic_increasing)

# 5) разные написания платформы
print("\nplatform:", sorted(events.platform.unique()))

событий после окна: 0.124
target
0.0    0.110
1.0    0.425
Name: доля событий после окна, dtype: float64

captcha_shown всего: 7928  из них внутри окна: 0



полных дублей строк: 4863
строки отсортированы по времени: False

platform: ['ANDROID', 'Android', 'IOS', 'WEB', 'Web', 'android', 'desktop', 'iOS', 'ios', 'iphone', 'web']


Что с этим делаю (`features.clean_events`):
- оставляю только события `window_start_ts <= event_ts < window_end_ts`. Заодно уходит и капча;
- удаляю полные дубли. У ботов и людей их доля одинаковая (~3%), это ошибка логирования;
- сортирую внутри куки по времени;
- `platform` привожу к `web / android / ios`;
- пропуски не заполняю. LightGBM работает с NaN, а сам пропуск информативен: например, нет курсора на web.

In [4]:
ev_clean = F.clean_events(events, meta)
print("событий после очистки:", len(ev_clean), "из", len(events))
print("у каждой куки есть события в окне:", ev_clean.cookie_id.nunique() == len(meta))
print(ev_clean.platform.value_counts())

событий после очистки: 283833 из 328905
у каждой куки есть события в окне: True
platform
web        158898
android    110556
ios         14379
Name: count, dtype: int64


## 3. Чем боты отличаются по поведению

Две самые сильные находки: интервалы между событиями и курсор.

In [5]:
e = ev_clean.merge(train[["cookie_id", "target"]], on="cookie_id")
e["gap"] = e.groupby("cookie_id").event_ts.diff().dt.total_seconds()

# интервалы внутри сессий (пауза до 30 минут): у ботов сдвинуты в 3-20 секунд
g = e[e.gap <= 1800]
bins = [0, 3, 10, 20, 30, 60, 120, 1800]
print("распределение интервалов между событиями (доли внутри класса):")
print(pd.crosstab(pd.cut(g.gap, bins, right=False), g.target, normalize="columns").round(3))

# курсор на web: у людей x равномерен по ширине 0-1920, у ботов прижат к левой части
w = e[e.platform == "web"]
print("\nдоля событий на web с координатами курсора:")
print(w.groupby("target").pointer_x.apply(lambda s: s.notna().mean()).round(3))
print("\nмедиана и 90-й перцентиль pointer_x:")
print(w.groupby("target").pointer_x.quantile([0.5, 0.9]).unstack().round(0))

распределение интервалов между событиями (доли внутри класса):

target           0      1
gap                      
[0, 3)       0.032  0.019
[3, 10)      0.098  0.367
[10, 20)     0.142  0.265
[20, 30)     0.128  0.117
[30, 60)     0.255  0.124
[60, 120)    0.201  0.046
[120, 1800)  0.143  0.061

доля событий на web с координатами курсора:
target
0    0.662
1    0.330
Name: pointer_x, dtype: float64

медиана и 90-й перцентиль pointer_x:
          0.5     0.9
target               
0       954.0  1726.0
1       668.0  1164.0


Что пробовал и что не сработало: конкретные запросы, категории и города. Доля ботов по ним почти одинаковая, 0.06–0.14. Ещё проверял регулярность пауз между сессиями и перебор `item_id` подряд, тоже ничего.

User-Agent сырой строкой не беру: 148 значений, модель бы их запомнила. Даже у HeadlessChrome или curl ботов только 25–50%, поэтому UA разбираю на признаки (headless, скрипт, приложение, версии), и они слабые.

## 4. Боты смотрят одни и те же объявления

На поведенческих признаках модель давала ~0.80, и дальше ничего не росло. Разбирая ошибки, увидел, что у ботов есть общий набор объявлений.

In [6]:
# популярность считаю по всем кукам (train и test, метки не нужны), долю — по train
pairs_all = ev_clean[ev_clean.item_id.notna()].drop_duplicates(["cookie_id", "item_id"])
n_cookies = pairs_all.groupby("item_id").cookie_id.nunique()
hot = n_cookies[n_cookies >= 10].index
pairs_tr = e[e.item_id.notna()].drop_duplicates(["cookie_id", "item_id"])
print("объявлений, которые смотрели 10+ кук:", len(hot), "из", len(n_cookies))
print("доля просмотров, приходящихся на такие объявления:")
print(pairs_tr.assign(hot=pairs_tr.item_id.isin(hot)).groupby("target").hot.mean().round(3))

# стабильно ли это во времени: доля ботов по объявлению за первую неделю
# должна предсказывать кук второй недели
early = pairs_tr[pairs_tr.window_start_ts < "2026-04-13"]
late = pairs_tr[pairs_tr.window_start_ts >= "2026-04-13"]
stats = early.groupby("item_id").target.agg(["sum", "count"])
late = late.join(stats, on="item_id").fillna({"sum": 0, "count": 0})
late["enc"] = (late["sum"] + 0.08 * 5) / (late["count"] + 5)
per_cookie = late.groupby("cookie_id").agg(enc=("enc", "mean"), target=("target", "first"))
print("\nROC-AUC: доля ботов по объявлениям 1-й недели -> куки 2-й недели:",
      round(roc_auc_score(per_cookie.target, per_cookie.enc), 3))

объявлений, которые смотрели 10+ кук: 497 из 52929
доля просмотров, приходящихся на такие объявления:
target
0    0.020
1    0.169
Name: hot, dtype: float64

ROC-AUC: доля ботов по объявлениям 1-й недели -> куки 2-й недели: 0.789


Отсюда три группы признаков. Все считаются только по тому, что известно к концу окна куки:

1. **Популярность** (`item_popularity_features`): сколько других кук смотрели те же объявления в тот же день или раньше, на 1000 кук. Меток не нужно. Окна кончаются в полночь, так что события других кук того же дня к концу окна уже есть.
2. **Доля ботов по объявлениям** (`item_target_features`): для куки из дня D только по меткам кук из дней **раньше D**. Своя метка и метки своего дня туда не попадают. Для test это вся разметка train.
3. **Скор соседей** (`neighbor_score_features` + `train.two_stage`). Первая модель ставит скор всем кукам, для кук из train он out-of-fold. Потом для каждой куки считается средний скор других кук, смотревших те же объявления. С этим признаком учится вторая модель. Это помогает в тесте, где неделя без меток.

## 5. Сборка признаков

In [7]:
# поведенческие признаки + популярность объявлений (без меток) для всех кук train и test
X_all, pairs = F.build_features(events, meta)
print("признаков без меток:", X_all.shape[1])
X_all.loc[train.cookie_id].head()

признаков без меток: 114


,cookie_age_days,hours_created_to_first,span_min,first_offset_h,n_sessions,max_session_events,max_session_dur_min,active_min,sess_min_gap_median,sess_max_fast_share,gap_min,gap_median,gap_max,frac_gap0,frac_gap_le3,...,ptr_step_median,ptr_step0_share,ua_nunique,ua_headless,ua_script,ua_app,ua_mobile,ua_platform_mismatch,chrome_ver,app_ver,os_ver,pop_mean,pop_max,pop_q75,pop_hot_share
cookie_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,135.603692,3256.012778,2.783333,1.524167,1,7,2.783333,2.783333,21.0,0.166667,1.0,21.0,84.0,0.0,0.166667,...,NaN,NaN,1,0,0,1,1,0.0,NaN,120.0,13.0,0.000000,0.000000,0.000000,0.000000
ck_7e4de46eeab82974,194.576111,4680.667222,657.933333,10.840556,7,13,9.650000,26.416667,27.0,0.400000,2.0,57.5,8511.0,0.0,0.025000,...,659.794665,0.0,1,0,0,0,0,0.0,116.0,NaN,NaN,0.068176,1.295337,0.000000,0.052632
ck_9320229ef6304522,32.994421,793.047778,608.183333,1.181667,7,10,32.616667,40.266667,11.0,0.750000,1.0,22.0,8350.0,0.0,0.028571,...,855.446249,0.0,1,0,0,0,0,0.0,116.0,NaN,NaN,0.204527,1.295337,0.000000,0.157895
ck_30ccd25bc1714ed9,0.546759,15.580556,287.016667,2.458333,3,10,6.833333,14.850000,41.0,0.222222,6.0,54.5,8604.0,0.0,0.000000,...,NaN,NaN,1,0,0,0,0,0.0,118.0,NaN,NaN,0.000000,0.000000,0.000000,0.000000
ck_a77c5f05948cdeef,94.837095,2284.223333,318.300000,8.133056,4,19,43.866667,52.916667,41.0,0.000000,22.0,52.0,7224.0,0.0,0.000000,...,599.205444,0.0,1,0,0,0,0,0.0,114.0,NaN,NaN,0.323834,1.295337,0.323834,0.250000


In [8]:
# признаки по меткам: для train это расширяющееся окно по дням,
# поэтому у первого дня истории ещё нет и там NaN
tgt = F.item_target_features(pairs, train.set_index("cookie_id").target)
first_day = train.cookie_id[train.window_start_ts == train.window_start_ts.min()]
print("у кук первого дня te_mean = NaN:", tgt.reindex(first_day).te_mean.isna().all())
tgt.describe().T.round(3)

у кук первого дня te_mean = NaN: True


,count,mean,std,min,25%,50%,75%,max
te_mean,14999.0,0.096,0.050,0.032,0.071,0.081,0.097,0.492
te_max,14999.0,0.183,0.134,0.032,0.082,0.089,0.237,0.672
te_q75,14999.0,0.101,0.067,0.032,0.075,0.082,0.084,0.530
te_bot_views,14999.0,2.797,8.677,0.000,0.000,0.000,2.000,182.000
te_bot_item_share,14999.0,0.125,0.215,0.000,0.000,0.000,0.167,1.000
te_seen_share,14999.0,0.629,0.308,0.000,0.429,0.667,0.875,1.000


## 6. Валидация

Тест (20–26 апреля) идёт сразу после train (6–19 апреля), поэтому валидация по времени:
- **4 фолда** по 2 дня (12–13, 14–15, 16–17, 18–19 апреля). Обучаюсь на всех днях до фолда;
- **7-дневный сплит «как тест»**: обучение 6–12, проверка 13–19 апреля.

Target-признаки в каждом фолде считаются заново только по меткам обучающей части. Метрика — `precision_at_recall` из `metric.py`.

Сравниваю по шагам:
1. baseline из quickstart;
2. только поведение;
3. плюс объявления;
4. плюс соседи.

Для скорости на валидации LightGBM с одним сидом, финальная модель усредняет пять. Ячейка идёт пару минут.

In [9]:
models = [
    ("baseline RF (2 признака)", T.baseline),
    ("LightGBM: поведение", T.behavior_only),
    ("LightGBM: + объявления", lambda *a: T.one_stage(*a, seeds=[0])),
    ("LightGBM: + объявления + соседи", lambda *a: T.two_stage(*a, seeds=[0])),
]
rows = []
for name, model in models:
    r = T.time_cv(model, X_all, pairs, train)
    rows.append({"модель": name, "P@R0.7 по фолдам": r["folds"], "среднее": r["mean_fold"],
                 "все фолды вместе": r["pooled"], "7-дневный сплит": r["horizon"],
                 "PR-AUC": r["pr_auc"], "ROC-AUC": r["roc_auc"]})
pd.DataFrame(rows).round(3)

,модель,P@R0.7 по фолдам,среднее,все фолды вместе,7-дневный сплит,PR-AUC,ROC-AUC
0,baseline RF (2 признака),"[0.087, 0.108, 0.088, 0.101]",0.096,0.095,0.093,0.205,0.609
1,LightGBM: поведение,"[0.808, 0.75, 0.832, 0.805]",0.799,0.767,0.755,0.800,0.938
2,LightGBM: + объявления,"[0.933, 0.896, 0.893, 0.845]",0.892,0.892,0.870,0.834,0.949
3,LightGBM: + объявления + соседи,"[0.926, 0.924, 0.9, 0.859]",0.902,0.895,0.883,0.839,0.948


Дополнительно проверял (в ноутбук не вошло, чтобы не раздувать):
- adversarial validation train vs test дала ROC-AUC 0.508, сдвига нет;
- без признаков курсора поведенческая модель падает с 0.80 до 0.59;
- RandomForest (0.68), ExtraTrees (0.77), HistGradientBoosting (0.79), логрег (0.64) и их ансамбли с LightGBM хуже одного LightGBM;
- перебор гиперпараметров LightGBM в пределах шума;
- K-fold target encoding вместо «только прошлые дни» дал то же качество. Оставил причинный вариант.

## 7. Финальная модель и сабмит

In [10]:
# обучение на всей разметке train (для test это строго прошлые дни), 5 сидов LightGBM
score = T.two_stage(X_all, pairs, train[["cookie_id", "target"]], test.cookie_id.values)
sub = pd.DataFrame({"cookie_id": test.cookie_id.values, "score": np.clip(score, 0, 1)})

# проверки формата: одна строка на куку из test, без пропусков
assert len(sub) == len(test) and sub.cookie_id.is_unique and sub.score.notna().all()
assert set(sub.cookie_id) == set(pd.read_csv("sample_submission.csv").cookie_id)
sub.to_csv("submission.csv", index=False)
print(len(sub), "строк сохранено в submission.csv")
sub.score.describe()

4909 строк сохранено в submission.csv


count    4909.000000
mean        0.080252
std         0.236056
min         0.000141
25%         0.001492
50%         0.003884
75%         0.013787
max         0.997709
Name: score, dtype: float64

## 8. Ограничения

- **Мобильные куки ранжируются хуже web**: P@R 0.55 против 0.93. На телефонах нет курсора, и часть мобильных ботов по событиям не отличается от людей. Похоже, разметка сделана по источнику трафика, а не по поведению.
- **Около recall 0.7 precision быстро падает**: 0.93 при recall 0.6, 0.77 при 0.8. Поэтому на закрытом тесте результат может отличаться на несколько пунктов.
- **Признаки по объявлениям работают, пока боты парсят один и тот же набор.** С курсором и UA то же: если боты начнут лучше имитировать браузер, эти признаки ослабнут. Модель надо переобучать на свежей разметке.